In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
%cd /content/drive/MyDrive/Intent-Classification-ML-Project/

/content/drive/.shortcut-targets-by-id/1zUMsAen5jnXf6Ycg2R8moTyjDKfN8zMZ/Intent-Classification-ML-Project


In [8]:
!ls

data  notebooks  README.md  requirements.txt  src


# Loading Stored Dataset with v1 features

In [9]:
import pandas as pd

# Paths should match what you used in 01
#features_path = "./data/rba_features_v1.parquet"
features_path = "./data/rule_baseline__v1+v2.parquet"

df = pd.read_parquet(features_path)

print("Loaded shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())  # first 40 cols, just to confirm

df.head()

Loaded shape: (300000, 70)

Columns:
['Login Timestamp', 'User ID', 'Round-Trip Time [ms]', 'IP Address', 'Country', 'Region', 'City', 'ASN', 'User Agent String', 'Browser Name and Version', 'OS Name and Version', 'Device Type', 'Login Successful', 'Is Attack IP', 'Is Account Takeover', 'browser', 'os', 'hour', 'dayofweek', 'is_new_device_for_user', 'is_new_ip_for_user', 'is_off_hours', 'failed_login', 'failures_last_5', 'failure_streak', 'failure_streak_capped', 'location', 'new_location_flag', 'new_asn_flag', 'device_fingerprint', 'valid_device', 'new_device_flag', 'device_change_rate', 'ts_sec', 'delta_sec', 'new_window', 'window_id', 'logins_5min', 'failure_flag', 'burst_failure_count', 'failure_rate', 'streak_reset', 'streak_id', 'failure_streak_length', 'location_freq', 'location_rarity', 'device_type_freq', 'device_type_rarity', 'asn_freq', 'asn_rarity', 'location_rarity_q', 'device_type_rarity_q', 'asn_rarity_q', 'user_offhour_rate', 'is_unusual_time_for_user', 'user_hour_std',

,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,...,rule_new_asn,rule_recent_failures,rule_risk_score,rule_risk_band,rule_risky_device_type,rule_high_risk_country,rule_risk_score_v2,rule_risk_band_v2,rule_decision_v2,pred_attack
0,2020-02-06 17:10:54.364,-9223287066183308537,541.0,84.209.76.159,no,oslo county,oslo,41164,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...,Chrome 69.0.3497.17.19,...,0,0,0,low,0,0,0,low,ALLOW,0
1,2020-02-06 19:52:41.530,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0,0,0,low,0,0,0,low,ALLOW,0
2,2020-02-06 20:55:19.627,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0,0,0,low,0,0,0,low,ALLOW,0
3,2020-02-05 21:03:20.657,-9223200578825105501,541.0,79.161.56.83,no,vestfold og telemark,holmestrand,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Chrome Mobile 81.0.4044.2033,...,0,0,0,low,0,0,0,low,ALLOW,0
4,2020-02-06 19:12:29.501,-9223199305075633823,541.0,79.161.86.86,no,-,-,29695,Mozilla/5.0 (Linux; U; Android 13.0; i phone X...,Opera Mobile 52.1.2254,...,0,0,0,low,0,0,0,low,ALLOW,0


In [10]:
# --- Split (same logic as LSTM) ---
df = df.sort_values("Login Timestamp").reset_index(drop=True)

t1 = df["Login Timestamp"].quantile(0.70)
t2 = df["Login Timestamp"].quantile(0.85)

def assign_split(ts):
    if ts <= t1:
        return "train"
    elif ts <= t2:
        return "val"
    else:
        return "test"

df["split"] = df["Login Timestamp"].apply(assign_split)

train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

print("Split counts:")
print(df["split"].value_counts())

print("\nAttack rate by split:")
print(df.groupby("split")["Is Attack IP"].mean())

print("\nShapes:", train_df.shape, val_df.shape, test_df.shape)

Split counts:
split
train    210000
val       45000
test      45000
Name: count, dtype: int64

Attack rate by split:
split
test     0.085578
train    0.094643
val      0.087244
Name: Is Attack IP, dtype: float64

Shapes: (210000, 71) (45000, 71) (45000, 71)


# Define the binary rule indicators (Iᵢ)


1. rule_unusual_time – off-hours & unusual for this user
2.	rule_off_hours – off-hours, but not unusual (user sometimes uses nights)
3.	rule_new_device – device changed since last login
4.	rule_new_asn – ASN changed since last login
5.	rule_recent_failures – at least 3 failures in last 5 logins



In [11]:
# 1) Rule: Unusual time for this user
# 1 if this login is off-hours AND user rarely uses off-hours (< 0.2),
# we already encoded this as is_unusual_time_for_user
df["rule_unusual_time"] = df["is_unusual_time_for_user"].astype(int)

# --- 2) Rule: Off-hours (but NOT already counted as unusual) ---
# Example: user sometimes logs at night, so it's off-hours but not rare for them
df["rule_off_hours"] = (
    (df["is_off_hours"] == 1) &
    (df["is_unusual_time_for_user"] == 0)
).astype(int)

# --- 3) Rule: New device since last login ---
df["rule_new_device"] = df["new_device_flag"].astype(int)

# --- 4) Rule: New ASN since last login ---
df["rule_new_asn"] = df["new_asn_flag"].astype(int)

# --- 5) Rule: Recent failures: 3 or more in previous 5 logins ---
# Make sure NaNs in failures_last_5 are treated as 0
failures_last_5_clean = df["failures_last_5"].fillna(0)
df["rule_recent_failures"] = (failures_last_5_clean >= 3).astype(int)

# Quick sanity check: show first few columns
df[[
    "is_off_hours",
    "is_unusual_time_for_user",
    "failures_last_5",
    "rule_unusual_time",
    "rule_off_hours",
    "rule_new_device",
    "rule_new_asn",
    "rule_recent_failures"
]].head()

,is_off_hours,is_unusual_time_for_user,failures_last_5,rule_unusual_time,rule_off_hours,rule_new_device,rule_new_asn,rule_recent_failures
0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,1,1,0
2,0,0,0,0,0,0,0,0
3,0,0,2,0,0,1,1,0
4,0,0,0,0,0,0,0,0


### For the above; validating the distributions just to make sure not all are 0's or 1's. We want evenly distributed!

In [12]:
rule_cols = [
    "rule_unusual_time",
    "rule_off_hours",
    "rule_new_device",
    "rule_new_asn",
    "rule_recent_failures",
]

for col in rule_cols:
    print(f"\n{col} value counts:")
    print(df[col].value_counts())


rule_unusual_time value counts:
rule_unusual_time
0    282476
1     17524
Name: count, dtype: int64

rule_off_hours value counts:
rule_off_hours
0    278226
1     21774
Name: count, dtype: int64

rule_new_device value counts:
rule_new_device
0    184842
1    115158
Name: count, dtype: int64

rule_new_asn value counts:
rule_new_asn
0    201323
1     98677
Name: count, dtype: int64

rule_recent_failures value counts:
rule_recent_failures
0    189779
1    110221
Name: count, dtype: int64


## Compute rule_risk_score


In [13]:
# Compute rule-based risk score as weighted sum of rule indicators

df["rule_risk_score"] = (
    2 * df["rule_unusual_time"]
    + 1 * df["rule_off_hours"]
    + 1 * df["rule_new_device"]
    + 1 * df["rule_new_asn"]
    + 1 * df["rule_recent_failures"]
)
# Refresh split dfs
train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

print("rule_risk_score summary:")
print(df["rule_risk_score"].describe())

print("\nCounts per rule_risk_score:")
print(df["rule_risk_score"].value_counts().sort_index())

rule_risk_score summary:
count    300000.000000
mean          1.269593
std           1.537271
min           0.000000
25%           0.000000
50%           0.000000
75%           3.000000
max           5.000000
Name: rule_risk_score, dtype: float64

Counts per rule_risk_score:
rule_risk_score
0    154037
1     37065
2     13555
3     78883
4      2246
5     14214
Name: count, dtype: int64



	•	Median = 0 → for at least 50% of logins, none of the 5 rules fired.
	•	These are “totally clean” from the baseline’s point of view.
	•	75th percentile = 3 → the top 25% of logins have score ≥ 3, meaning:
	•	multiple rules triggered together (e.g., off-hours + new device + failures).
	•	Mean ~1.27, std ~1.54 → most logins are low-to-moderate risk; a smaller group has piled-up risk.

This is good:
We don’t want everything to look risky, but we do want a clear separation between “no rule triggered” and “multiple rules triggered”.



1.   Score = 0 → 154,037 logins (~51.3%)

	•	None of the rules fired:
	•	Not off-hours
	•	Not unusual time
	•	No new device
	•	No new ASN
	•	No heavy recent failures

These are your “baseline-normal” logins.
This is exactly what we want: about half the data looks totally unremarkable.

⸻

2. Score = 1 → 37,065 logins (~12.4%)

	•	Exactly one mild rule fired:

	•	Example: just off-hours but not unusual for that user,
	•	or just new device,

	•	or just new ASN,

	•	or just 3+ recent failures.

These are slightly risky but not alarming.
Good place to label as low risk but “watch”.

⸻
3. Score = 2 → 13,555 logins (~4.5%)

	•	Either:
	•	Unusual time only (2 points), or
	•	two mild rules (e.g., new device + new ASN).

This is your first “this looks genuinely suspicious” layer.

Remember: unusual time on its own got +2 because in EDA it had ~2× attack rate.

⸻

4.  Score = 3 → 78,883 logins (~26.3%)

	•	Examples:
	•	unusual time (2) + new device (1)
	•	unusual time (2) + recent failures (1)
	•	or 3 different +1 rules together (off-hours + new ASN + recent failures, etc.)

This is a big chunk of the dataset (~1/4) where multiple risk factors align.

This is the region where “step-up auth” is very natural: user might still be legit, but we want an extra check.

⸻

5. Score = 4 → 2,246 logins (~0.75%)

	•	Example:
	•	unusual time (2) + any two of {off-hours, new device, new ASN, recent failures}
	•	or four mild rules firing at the same time.

Very clustered risk; these are strong candidates for:

	•	Step-up auth at minimum, often block / review.

⸻

6. Score = 5 → 14,214 logins (~4.7%)

	•	This is near-max risk:
	•	E.g., unusual time (2) + three other risk factors (off-hours + new device + recent failures), etc.

These are your top-risk logins.
In a real system, many of these would probably be outright blocked or heavily challenged.

# Potential extra rule signals (for v2 baseline later)

	1.	Device type risk
	•	From EDA: Device Type = bot and unknown had very high attack rates.
	•	Rule idea:
	•	rule_risky_device_type = 1 if device_type in {bot, unknown}

	2.	Global rarity
	•	asn_rarity or location_rarity:
	•	Very rare ASNs or locations might be suspicious.
	•	Rule idea:
	•	rule_rare_asn = 1 if asn_rarity > some_threshold
	•	rule_rare_location = 1 if location_rarity > some_threshold

	3.	Stronger failure pattern
	•	You already have failure_streak_capped and burst_failure_count.
	•	Rule idea:
	•	rule_long_streak = 1 if failure_streak_capped == 5
	•	rule_burst_failures = 1 if burst_failure_count >= X

	4.	Country-based risk
	•	Some countries in EDA (e.g., small set with 50%+ attack rate) were much riskier.
	•	Rule idea:
	•	rule_high_risk_country = 1 if Country in {list_of_very_high_attack_rate_countries}

# Mapping rule_risk_score → rule_risk_band (low / medium / high)

In [14]:
def map_rule_band(score: int) -> str:
    if score <= 1:
        return "low"
    elif score <= 3:
        return "medium"
    else:
        return "high"

df["rule_risk_band"] = df["rule_risk_score"].apply(map_rule_band)

print("Risk band counts:")
print(df["rule_risk_band"].value_counts())

Risk band counts:
rule_risk_band
low       191102
medium     92438
high       16460
Name: count, dtype: int64


### So we found based on Risk Score we found earlier

###	•	Low risk → ~63.7% of logins
###	•	Medium risk → ~30.8%
###	•	High risk → ~5.5%

# Checking Check attack rate per band using Is Attack IP

In [15]:
label = "Is Attack IP"

print("\nAttack rate by rule_risk_band:")
print(df.groupby("rule_risk_band")[label].mean())

print("\nCrosstab of rule_risk_band vs label:")
print(pd.crosstab(df["rule_risk_band"], df[label]))


Attack rate by rule_risk_band:
rule_risk_band
high      0.183111
low       0.075337
medium    0.110788
Name: Is Attack IP, dtype: float64

Crosstab of rule_risk_band vs label:
Is Attack IP     False  True 
rule_risk_band               
high             13446   3014
low             176705  14397
medium           82197  10241


In [16]:
import numpy as np
from sklearn.metrics import precision_recall_curve

def tune_threshold(val_score, y_val, min_recall=0.70):
    prec, rec, thr = precision_recall_curve(y_val, val_score)

    prec2, rec2 = prec[:-1], rec[:-1]
    f1 = (2 * prec2 * rec2) / (prec2 + rec2 + 1e-12)

    mask = rec2 >= min_recall

    if mask.any():
        best_idx = int(np.argmax(np.where(mask, f1, -1)))
    else:
        best_idx = int(np.argmax(f1))

    best_thr = float(thr[best_idx])

    print("Min recall constraint:", min_recall)
    print("Best threshold:", best_thr)
    print("VAL Precision:", float(prec2[best_idx]))
    print("VAL Recall   :", float(rec2[best_idx]))
    print("VAL F1       :", float(f1[best_idx]))

    return best_thr

In [17]:
y_val = val_df["Is Attack IP"].astype(int).values
y_test = test_df["Is Attack IP"].astype(int).values

In [20]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

def evaluate_binary_classifier(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_score = np.asarray(y_score).ravel()

    y_pred = (y_score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_score)

    specificity = tn / (tn + fp + 1e-12)
    fpr = 1 - specificity
    fnr = fn / (fn + tp + 1e-12)

    print(f"\nThreshold: {threshold:.4f}")
    print(f"Confusion Matrix (TN, FP, FN, TP): {tn}, {fp}, {fn}, {tp}")
    print(f"Precision  : {precision:.4f}")
    print(f"Recall     : {recall:.4f}")
    print(f"F1-score   : {f1:.4f}")
    print(f"FPR        : {fpr:.4f}")
    print(f"FNR        : {fnr:.4f}")
    print(f"AUC        : {auc:.4f}")

    return {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr),
        "fnr": float(fnr),
        "auc": float(auc),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

In [21]:
print("\n=== RULE V1 (VAL tuning) ===")

val_score_v1 = val_df["rule_risk_score"].values
test_score_v1 = test_df["rule_risk_score"].values

best_v1_thr = tune_threshold(val_score_v1, y_val, min_recall=0.70)

print("\n=== RULE V1 (TEST at best_v1_thr) ===")
evaluate_binary_classifier(y_test, test_score_v1, threshold=best_v1_thr)


=== RULE V1 (VAL tuning) ===
Min recall constraint: 0.7
Best threshold: 0.0
VAL Precision: 0.08724444444444444
VAL Recall   : 1.0
VAL F1       : 0.16048726648393036

=== RULE V1 (TEST at best_v1_thr) ===

Threshold: 0.0000
Confusion Matrix (TN, FP, FN, TP): 0, 41149, 0, 3851
Precision  : 0.0856
Recall     : 1.0000
F1-score   : 0.1577
FPR        : 1.0000
FNR        : 0.0000
AUC        : 0.6186


{'threshold': 0.0,
 'precision': 0.08557777777777778,
 'recall': 1.0,
 'f1': 0.15766309799185277,
 'fpr': 1.0,
 'fnr': 0.0,
 'auc': 0.6185845065818056,
 'tn': 0,
 'fp': 41149,
 'fn': 0,
 'tp': 3851}

## Result

Out of all 300,000 logins, about 9.2% are attacks. That's your "before we do anything" number.

#### 1. LOW risk band (the "probably fine" line)
```
191,102 people were put here

7.5% turned out to be attackers

Lower than the 9.2% global rate → the rules successfully moved suspicious people OUT of this group

But 14,397 real attackers still snuck in here
```

#### 2. MEDIUM risk band (the "hmm, let me check you" line)
```
92,438 people here

11.1% are attackers

Higher than global rate → good, the rules are pulling more suspicious people into this bucket
```

#### 3. HIGH risk band (the "step aside" line)



```
Only 16,460 people here

18.3% are attackers

Almost 2× the global rate → your rules are successfully concentrating the most suspicious logins here

```

Our rules work in the right direction —

as risk band goes up, attack rate goes up.

Low → 7.5%, Medium → 11.1%, High → 18.3%,

 compared to a global average of 9.2%. The system successfully pushes risky logins upward.

Risk is monotonic:

	•	High band is clearly high risk (~18.3% vs global 9.2%).
	•	Band distribution (63% low, 31% medium, 5.5% high) is realistic for:
	•	Allow most logins,
	•	Step-up a subset,
	•	Hard-check a small tail.

We built a weighted scoring system where each rule contributes points to a total risk score. No single rule makes the final decision — the combined score determines whether a login is allowed, stepped up, or blocked.

# We implemented a rule-based baseline v1 that uses five interpretable signals:



```
1.   unusual login time,
2.   off-hours access,
3.   new device, new ASN,
4.   recent failure bursts.
Each rule contributes a fixed number of points to a rule risk score, which is then mapped into low, medium, and high risk bands.
```

In our experiments on 300,000 login events, the overall attack rate is about 9.2%. In the low, medium, and high risk bands, the attack rates are approximately 7.5%, 11.1%, and 18.3% respectively.

This shows that the rule-based baseline successfully concentrates higher-risk logins in the medium and high bands, but also that a significant portion of attacks still fall into the low band. This motivates the need for a learned intent classifier that can exploit richer temporal and contextual patterns than fixed rules.

# ** v2 baseline**

Why not using IP address -

* Super high cardinality There can be thousands of distinct IPs. If I write rules like

if IP == X then high risk, those rules won’t generalize to new IPs you’ve never seen.

In this dataset we do know Is Attack IP. It would be trivial (not much useful) to make a rule that says:

 ***if this IP was ever marked attack before → risk=high.
That’s basically leaking the label into the feature. In the real world, you don’t know the future label yet.***

More meaningful abstractions exist Instead of raw IP, we want:

	•	per-user IP behavior → is_new_ip_for_user
	•	IP’s network / ASN → ASN, new_asn_flag, asn_rarity
	•	geo context → Country/Region/City, location_rarity

We avoid using raw IP addresses as rules, because they don’t generalize and basically act like a blacklist. Instead, we derive higher-level behavior features like ‘new ASN’, ‘new location’, and IP-based rarity. We also tested is_new_ip_for_user and found that, in this dataset, new IPs are less associated with attack IPs, likely because attackers re-use a small set of IPs while legitimate users move across networks. So we keep is_new_ip_for_user as an input feature for the ML model rather than using it as a simple risk-raising rule in the baseline.”


### Inspecting attack rate for Device Type & Country

In [22]:
import pandas as pd

label = "Is Attack IP"

# --- A) Device Type vs attack rate ---
print("Attack rate by Device Type:")
device_stats = (
    df.groupby("Device Type")[label]
      .agg(["count", "mean"])
      .rename(columns={"mean": "attack_rate"})
      .sort_values("attack_rate", ascending=False)
)
print(device_stats.head(10))  # top 10 risky device types

# --- B) Country vs attack rate (only if enough samples) ---
print("\nAttack rate by Country (only countries with >= 200 events):")
country_stats = (
    df.groupby("Country")[label]
      .agg(["count", "mean"])
      .rename(columns={"mean": "attack_rate"})
)
country_stats_filtered = country_stats[country_stats["count"] >= 200] \
    .sort_values("attack_rate", ascending=False)
print(country_stats_filtered.head(15))  # top 15 risky countries

Attack rate by Device Type:
              count  attack_rate
Device Type                     
bot             234     0.773504
unknown         167     0.323353
mobile       212206     0.117080
desktop       78276     0.029779
tablet         9117     0.026434

Attack rate by Country (only countries with >= 200 events):
         count  attack_rate
Country                    
ru         313     0.549521
cl         442     0.533937
ro        1733     0.466821
us       64532     0.310187
vn         225     0.288889
pk         203     0.266010
ph         474     0.175105
tr        1660     0.137952
id        4292     0.129077
au        3809     0.115516
pl       17398     0.098632
dk         342     0.087719
ca         669     0.074738
it         643     0.062208
nl         963     0.059190



We Found:

#### Device Type


```
	•	bot → insanely risky (3/4 are attack IPs)
	•	unknown → moderate risk (about 1/3 attacks) [Other device types]
	•	mobile → slightly risky relative to global (~11.7% vs ~9.2%)
	•	desktop/tablet → low risk.

So it makes sense to treat:

Device Type in {bot, unknown} as risky device types.
```

#### Location Type
```
	• ru, cl, ro → extremely high attack rates (≈ 0.47–0.55).
	•	us, vn, pk → clearly above global rate (~0.27–0.31).
	•	Others start dropping closer to / below overall 0.092.

So a simple, data-driven heuristic:

Treat countries with attack_rate >= 0.25 as high-risk countries.
```

#### rule_risky_device_type

#### rule_high_risk_country

In [23]:
label = "Is Attack IP"

# --- 1) Rule: risky device types (bot, unknown) ---
df["rule_risky_device_type"] = df["Device Type"].isin(["bot", "unknown"]).astype(int)

print("rule_risky_device_type value counts:")
print(df["rule_risky_device_type"].value_counts())

print("\nAttack rate by rule_risky_device_type:")
print(df.groupby("rule_risky_device_type")[label].mean())


# --- 2) Rule: high-risk countries ---

# You can either hardcode based on your stats:
# high_risk_countries = ["ru", "cl", "ro", "us", "vn", "pk"]

# Or compute them programmatically (optional):
country_stats = (
    df.groupby("Country")[label]
      .agg(["count", "mean"])
      .rename(columns={"mean": "attack_rate"})
)
high_risk_countries = country_stats[
    (country_stats["count"] >= 200) & (country_stats["attack_rate"] >= 0.25)
].index.tolist()
print("High-risk countries (>=200 events, attack_rate>=0.25):", high_risk_countries)

df["rule_high_risk_country"] = df["Country"].isin(high_risk_countries).astype(int)

print("\nrule_high_risk_country value counts:")
print(df["rule_high_risk_country"].value_counts())

print("\nAttack rate by rule_high_risk_country:")
print(df.groupby("rule_high_risk_country")[label].mean())

rule_risky_device_type value counts:
rule_risky_device_type
0    299599
1       401
Name: count, dtype: int64

Attack rate by rule_risky_device_type:
rule_risky_device_type
0    0.091512
1    0.586035
Name: Is Attack IP, dtype: float64
High-risk countries (>=200 events, attack_rate>=0.25): ['cl', 'pk', 'ro', 'ru', 'us', 'vn']

rule_high_risk_country value counts:
rule_high_risk_country
0    232552
1     67448
Name: count, dtype: int64

Attack rate by rule_high_risk_country:
rule_high_risk_country
0    0.027086
1    0.316585
Name: Is Attack IP, dtype: float64


We already lnow:

	•	Risky device type (bot/unknown):
	•	Very rare (~0.13% of logins) but extremely dangerous.
	•	Perfect as a high-weight rule in v2.

	•	High-risk country (ru, cl, ro, us, vn, pk):
	•	~22.5% of all events.
	•	Attack rate is more than 10× higher than non-high-risk countries (31.7% vs 2.7%).
	•	This is a major risk driver.


#Statistics v1:

Below are stats for v1

In [24]:
def summarize_baseline(df, label_col, score_col, band_col):
    total = len(df)
    total_attacks = df[label_col].sum()
    global_rate = total_attacks / total

    print(f"Total events: {total}")
    print(f"Total attacks: {int(total_attacks)}")
    print(f"Global attack rate: {global_rate:.4f}")
    print("-" * 50)

    # --- Band-level stats ---
    band_stats = (
        df.groupby(band_col)[label_col]
          .agg(["count", "mean"])
          .rename(columns={"count": "events", "mean": "attack_rate"})
    )

    band_stats["events_pct"] = band_stats["events"] / total
    band_stats["attacks"] = (
        df.groupby(band_col)[label_col].sum().astype(int)
    )
    band_stats["attacks_pct_of_all"] = band_stats["attacks"] / total_attacks
    band_stats["attack_rate_lift"] = band_stats["attack_rate"] / global_rate

    print("Band-level stats:")
    display(band_stats.sort_index())

    # --- Rule-level stats ---
    rule_cols = [
        "rule_unusual_time",
        "rule_off_hours",
        "rule_new_device",
        "rule_new_asn",
        "rule_recent_failures",
        # later we’ll add:
        # "rule_risky_device_type",
        # "rule_high_risk_country",
    ]

    rows = []
    for col in rule_cols:
        trig = df[df[col] == 1]
        notrig = df[df[col] == 0]

        rows.append({
            "rule": col,
            "triggered_events": len(trig),
            "triggered_pct": len(trig) / total,
            "attack_rate_when_triggered": trig[label_col].mean(),
            "attack_rate_when_not_triggered": notrig[label_col].mean(),
        })

    rule_stats = pd.DataFrame(rows)
    print("\nRule-level stats:")
    display(rule_stats)

    return band_stats, rule_stats

In [25]:
label_col = "Is Attack IP"
score_col = "rule_risk_score"      # will be rule_risk_score_v2 later
band_col = "rule_risk_band"        # will be rule_risk_band_v2 later

band_stats_v1, rule_stats_v1 = summarize_baseline(df, label_col, score_col, band_col)

Total events: 300000
Total attacks: 27652
Global attack rate: 0.0922
--------------------------------------------------
Band-level stats:


,events,attack_rate,events_pct,attacks,attacks_pct_of_all,attack_rate_lift
rule_risk_band,,,,,,
high,16460,0.183111,0.054867,3014,0.108998,1.986589
low,191102,0.075337,0.637007,14397,0.520650,0.817338
medium,92438,0.110788,0.308127,10241,0.370353,1.201950



Rule-level stats:


,rule,triggered_events,triggered_pct,attack_rate_when_triggered,attack_rate_when_not_triggered
0,rule_unusual_time,17524,0.058413,0.185916,0.086358
1,rule_off_hours,21774,0.072580,0.141637,0.088302
2,rule_new_device,115158,0.383860,0.113505,0.078884
3,rule_new_asn,98677,0.328923,0.110928,0.082981
4,rule_recent_failures,110221,0.367403,0.125366,0.072895


## V1 Baseline — Band-Level Results

**Global context:** Out of 300,000 login events, 27,652 are attacks —
giving us a global attack rate of **9.2%**. This is our "do-nothing" benchmark.
Everything we evaluate is compared against this number.

| Band   | Events | % of Traffic | Attack Rate | vs Global (Lift) | Attacks Captured |
|--------|--------|--------------|-------------|------------------|-----------------|
| Low    | 191,102 | 63.7%       | 7.5%        | 0.82× (↓ safer) | 52.1% of all attacks |
| Medium | 92,438  | 30.8%       | 11.1%       | 1.20× (↑ riskier) | 37.0% of all attacks |
| High   | 16,460  | 5.5%        | 18.3%       | 1.99× (↑↑ dangerous) | 10.9% of all attacks |

### How to read this table

**"Events %"** — How much of your traffic lands in each bucket.
63.7% of logins are considered low risk. Only 5.5% hit the high-risk band.
This is intentional — you don't want to challenge every user.

**"Attack Rate"** — Out of logins in that band, what fraction are real attacks.
The key story: low → medium → high shows a clean monotonic increase (7.5% → 11.1% → 18.3%).
The rules are working in the right direction.

**"Lift"** — Attack rate in this band ÷ global attack rate (9.2%).
- Lift < 1 means this band is *safer* than average (low band = 0.82×)
- Lift > 1 means this band is *riskier* than average (high band ≈ 2×)
- High band lift of ~2 means: a login flagged as high risk is **twice as likely**
  to be an attack compared to a random login.

**"Attacks Captured"** — What % of ALL attacks fall into this band.
The low band still captures 52% of attacks — meaning more than half of attackers
look "normal" by these rules alone. This is the core limitation of the v1 baseline
and motivates moving to a learned ML model.

## V1 Baseline — Rule-Level Results

Each row below shows one rule signal, how often it fires,
and how much it shifts the attack rate.

| Rule | Fires on | Attack rate WHEN triggered | Attack rate when NOT triggered | Signal Strength |
|------|----------|---------------------------|-------------------------------|----------------|
| rule_unusual_time   | 5.8%  | **18.6%** | 8.6% | Strong |
| rule_off_hours      | 7.3%  | **14.2%** | 8.8% | Moderate |
| rule_new_device     | 38.4% | **11.4%** | 7.9% | Mild |
| rule_new_asn        | 32.9% | **11.1%** | 8.3% | Mild |
| rule_recent_failures| 36.7% | **12.5%** | 7.3% | Moderate |

### How to read this table

**"Fires on %"** — How frequently this rule triggers across all 300K logins.
- `rule_unusual_time` is very selective — only 5.8% of logins trigger it.
- `rule_new_device` is broad — fires on 38% of all logins.

**"Attack rate when triggered vs not"** — This is the key column.
Every single rule shows a higher attack rate when it fires vs when it doesn't.
That validates each rule is a real signal, not noise.

**Rule-by-rule interpretation:**

- **Unusual time (18.6% vs 8.6%):** Logins at unusual hours for that specific user
  are more than 2× as likely to be attacks. Strongest individual signal.

- **Off-hours (14.2% vs 8.8%):** Even when it's not unusual for the user personally,
  off-hours logins carry elevated risk. Slightly weaker than unusual_time
  because we already excluded the "truly unusual" users.

- **New device (11.4% vs 7.9%):** A device change is a moderate signal.
  Fires often (38% of logins), so it alone isn't a strong alarm —
  but combined with other rules, it adds meaningful weight.

- **New ASN (11.1% vs 8.3%):** Network/provider changes carry similar mild risk
  to device changes. Useful as a corroborating signal.

- **Recent failures (12.5% vs 7.3%):** 3+ failures in the last 5 logins is
  a meaningful red flag — nearly 1.7× the rate vs clean login histories.
  Brute-force or credential stuffing often shows up here.

### Bottom line for v1
All 5 rules are individually validated. No rule is noise.  
The weakness isn't the rules themselves — it's that rules applied independently  
can't capture *combinations* of weak signals the way a trained model can.  
That gap is what the ML classifier in the next notebook addresses.

The rule-based baseline successfully concentrates higher-risk logins into the high-risk band, where the attack rate is almost twice the global average (18.3% vs 9.2%), while the low-risk band shows a reduced attack rate of 7.5% over 63% of all logins.

All five rule signals show a higher attack rate when triggered compared to when they are inactive. In particular, unusual login times and recent failure bursts nearly double the attack rate, confirming that time-based anomalies and error patterns are strong intent signals.

### Compute rule_risk_score_v2

In [26]:
# --- Rule-Based Risk Score V2 (with new rules) ---

df["rule_risk_score_v2"] = (
    2 * df["rule_unusual_time"]
    + 1 * df["rule_off_hours"]
    + 1 * df["rule_new_device"]
    + 1 * df["rule_new_asn"]
    + 1 * df["rule_recent_failures"]
    + 3 * df["rule_risky_device_type"]
    + 2 * df["rule_high_risk_country"]
)

print("rule_risk_score_v2 summary:")
print(df["rule_risk_score_v2"].describe())

print("\nCounts per rule_risk_score_v2:")
print(df["rule_risk_score_v2"].value_counts().sort_index())

rule_risk_score_v2 summary:
count    300000.000000
mean          1.723257
std           1.868419
min           0.000000
25%           0.000000
50%           1.000000
75%           3.000000
max          10.000000
Name: rule_risk_score_v2, dtype: float64

Counts per rule_risk_score_v2:
rule_risk_score_v2
0     128760
1      27444
2      34446
3      67576
4       4600
5      29283
6       2249
7       5538
8        101
10         3
Name: count, dtype: int64


## V2 Risk Score — Understanding the Distribution

Before we map scores into bands, let's understand what the score distribution
is actually telling us.

Quick reminder of how v2 scoring works:
- rule_unusual_time    → +2 points
- rule_off_hours       → +1 point
- rule_new_device      → +1 point
- rule_new_asn         → +1 point
- rule_recent_failures → +1 point
- rule_risky_device    → +3 points  ← new in v2
- rule_high_risk_country → +2 points ← new in v2

Maximum possible score = 11 (but practically we see up to 10)

---

### What the Summary Stats Tell Us

- **Mean = 1.72** — The average login only triggers about 1-2 rules.
  Most logins look pretty normal.

- **Median = 1** — Half of all logins score 1 or below.
  The "typical" user barely triggers any rules.

- **75th percentile = 3** — Only the top 25% of logins score 3 or higher.
  These are the ones where multiple risk signals pile up together.

- **Max = 10** — A small handful of logins triggered almost every rule at once.
  These are your most suspicious cases.

---

### Score-by-Score Breakdown

| Score | Count   | % of Traffic | What it likely means |
|-------|---------|--------------|----------------------|
| 0     | 128,760 | 42.9%        | Zero rules fired. Clean login, no red flags at all. |
| 1     | 27,444  | 9.1%         | One mild rule fired (e.g. new device, or new ASN). Probably fine. |
| 2     | 34,446  | 11.5%        | Either unusual time alone (worth +2), or two mild rules together. Starting to look suspicious. |
| 3     | 67,576  | 22.5%        | High-risk country (+2) + one mild rule, or unusual time + new device. Clearly elevated. |
| 4     | 4,600   | 1.5%         | Multiple signals stacking up. Strong candidate for step-up auth. |
| 5     | 29,283  | 9.8%         | High-risk country + unusual time + another rule. Seriously suspicious. |
| 6     | 2,249   | 0.75%        | Risky device type (+3) + other signals. Very dangerous zone. |
| 7     | 5,538   | 1.8%         | Near max risk. Almost every major rule fired. |
| 8     | 101     | ~0%          | Extremely rare. Basically every rule firing at once. |
| 10    | 3       | ~0%          | The worst 3 logins in 300,000. Every possible rule triggered. |

---

### The Natural Split

Looking at the counts, three clusters emerge naturally:

**Score 0–2 → ~63.5% of traffic 128,760 + 27,444 + 34,446 ≈ 190,650 events**

The majority of logins. Either nothing fired, or only weak/single signals.
These are your ALLOW candidates.

**Score 3–5 → ~33.8% of traffic 67,576 + 4,600 + 29,283 ≈ 101,459 events**

Multiple risk factors aligning. Not necessarily an attack, but worth friction.
These are your STEP-UP candidates — ask for MFA, verify identity.

**Score 6+ → ~2.6% of traffic 2,249 + 5,538 + 101 + 3 ≈ 7,891 events**

Rare but dangerous. Risky device types and heavily stacked signals.
These are your BLOCK candidates.

---

### Why the Big Jump at Score 3?

You'll notice score 3 has 67,576 events — way more than score 2 or 4.

This is because **rule_high_risk_country (+2) + any single mild rule (+1) = 3.**
And high-risk countries cover about 22.5% of all logins.
So a huge chunk of traffic naturally lands exactly at score 3 from that combination alone.

This is actually useful information — it tells us that geography is a dominant
risk driver in this dataset, which is why we gave it a weight of +2 in v2.

#### Map v2 score → v2 bands

In [27]:
def map_rule_band_v2(score: int) -> str:
    if score <= 2:
        return "low"
    elif score <= 5:
        return "medium"
    else:
        return "high"

df["rule_risk_band_v2"] = df["rule_risk_score_v2"].apply(map_rule_band_v2)

print("V2 Risk band counts:")
print(df["rule_risk_band_v2"].value_counts())

V2 Risk band counts:
rule_risk_band_v2
low       190650
medium    101459
high        7891
Name: count, dtype: int64


## V2 Risk Band Distribution

After mapping scores into bands:

| Band   | Logins  | % of Traffic | Decision |
|--------|---------|--------------|----------|
| Low    | 190,650 | 63.5%        | ALLOW    |
| Medium | 101,459 | 33.8%        | STEP-UP  |
| High   | 7,891   | 2.6%         | BLOCK    |

This is a healthy distribution for a real-world auth system.

- **63.5% of users get in with zero friction** — good for user experience
- **33.8% get a small challenge** (MFA, verify identity) — reasonable middle ground
- **Only 2.6% are blocked** — we're not being trigger-happy with blocks

The high band shrunk compared to v1 (was 5.5%, now 2.6%), but it's much more
precise — meaning the logins we do block are far more likely to actually be attacks.
Quality over quantity.

In [28]:
def map_decision_from_band(band: str) -> str:
    if band == "low":
        return "ALLOW"
    elif band == "medium":
        return "STEP_UP"
    else:
        return "BLOCK"

df["rule_decision_v2"] = df["rule_risk_band_v2"].apply(map_decision_from_band)

print("Decision counts (v2):")
print(df["rule_decision_v2"].value_counts())

Decision counts (v2):
rule_decision_v2
ALLOW      190650
STEP_UP    101459
BLOCK        7891
Name: count, dtype: int64


# Statistics v2

Reuse the stats helper for v2

In [29]:
def summarize_baseline(df, label_col, score_col, band_col):
    total = len(df)
    total_attacks = df[label_col].sum()
    global_rate = total_attacks / total

    print(f"Total events: {total}")
    print(f"Total attacks: {int(total_attacks)}")
    print(f"Global attack rate: {global_rate:.4f}")
    print("-" * 50)

    # --- Band-level stats ---
    band_stats = (
        df.groupby(band_col)[label_col]
          .agg(["count", "mean"])
          .rename(columns={"count": "events", "mean": "attack_rate"})
    )

    band_stats["events_pct"] = band_stats["events"] / total
    band_stats["attacks"] = (
        df.groupby(band_col)[label_col].sum().astype(int)
    )
    band_stats["attacks_pct_of_all"] = band_stats["attacks"] / total_attacks
    band_stats["attack_rate_lift"] = band_stats["attack_rate"] / global_rate

    print("Band-level stats:")
    display(band_stats.sort_index())

    # --- Rule-level stats ---
    rule_cols = [
        "rule_unusual_time",
        "rule_off_hours",
        "rule_new_device",
        "rule_new_asn",
        "rule_recent_failures",
        "rule_risky_device_type",
        "rule_high_risk_country",
    ]
    rows = []
    for col in rule_cols:
        trig = df[df[col] == 1]
        notrig = df[df[col] == 0]

        rows.append({
            "rule": col,
            "triggered_events": len(trig),
            "triggered_pct": len(trig) / total,
            "attack_rate_when_triggered": trig[label_col].mean(),
            "attack_rate_when_not_triggered": notrig[label_col].mean(),
        })

    rule_stats = pd.DataFrame(rows)
    print("\nRule-level stats:")
    display(rule_stats)

    return band_stats, rule_stats

In [30]:
label_col = "Is Attack IP"
score_col_v2 = "rule_risk_score_v2"
band_col_v2 = "rule_risk_band_v2"

band_stats_v2, rule_stats_v2 = summarize_baseline(df, label_col, score_col_v2, band_col_v2)

Total events: 300000
Total attacks: 27652
Global attack rate: 0.0922
--------------------------------------------------
Band-level stats:


,events,attack_rate,events_pct,attacks,attacks_pct_of_all,attack_rate_lift
rule_risk_band_v2,,,,,,
high,7891,0.338107,0.026303,2668,0.096485,3.668162
low,190650,0.059785,0.635500,11398,0.412194,0.648614
medium,101459,0.133906,0.338197,13586,0.491321,1.452766



Rule-level stats:


,rule,triggered_events,triggered_pct,attack_rate_when_triggered,attack_rate_when_not_triggered
0,rule_unusual_time,17524,0.058413,0.185916,0.086358
1,rule_off_hours,21774,0.072580,0.141637,0.088302
2,rule_new_device,115158,0.383860,0.113505,0.078884
3,rule_new_asn,98677,0.328923,0.110928,0.082981
4,rule_recent_failures,110221,0.367403,0.125366,0.072895
5,rule_risky_device_type,401,0.001337,0.586035,0.091512
6,rule_high_risk_country,67448,0.224827,0.316585,0.027086


In [31]:
print("\n=== RULE V2 (VAL tuning) ===")
val_score_v2 = val_df["rule_risk_score_v2"].values
test_score_v2 = test_df["rule_risk_score_v2"].values

best_v2_thr = tune_threshold(val_score_v2, y_val, min_recall=0.70)

print("\n=== RULE V2 (TEST at best_v2_thr) ===")
evaluate_binary_classifier(y_test, test_score_v2, threshold=best_v2_thr)


=== RULE V2 (VAL tuning) ===
Min recall constraint: 0.7
Best threshold: 2.0
VAL Precision: 0.1634280257818095
VAL Recall   : 0.8718797758532858
VAL F1       : 0.27526034337153255

=== RULE V2 (TEST at best_v2_thr) ===

Threshold: 2.0000
Confusion Matrix (TN, FP, FN, TP): 23488, 17661, 505, 3346
Precision  : 0.1593
Recall     : 0.8689
F1-score   : 0.2692
FPR        : 0.4292
FNR        : 0.1311
AUC        : 0.7529


{'threshold': 2.0,
 'precision': 0.15928023992002666,
 'recall': 0.8688652298104389,
 'f1': 0.2692091077319173,
 'fpr': 0.42919633526938683,
 'fnr': 0.13113477018956113,
 'auc': 0.7528781455116729,
 'tn': 23488,
 'fp': 17661,
 'fn': 505,
 'tp': 3346}

## V2 Baseline — Full Results

Global attack rate is 9.2% — everything below is measured against this number.

---

### Band-Level Stats

| Band   | Events  | % Traffic | Attack Rate | Lift  | Attacks Captured |
|--------|---------|-----------|-------------|-------|-----------------|
| Low    | 190,650 | 63.5%     | 6.0%        | 0.65× | 41.2% of all attacks |
| Medium | 101,459 | 33.8%     | 13.4%       | 1.45× | 49.1% of all attacks |
| High   | 7,891   | 2.6%      | 33.8%       | 3.67× | 9.6% of all attacks |

**How to read lift:**
- Low band lift 0.65 → logins here are 35% *less likely* to be attacks than average. Safer zone.
- High band lift 3.67 → logins here are almost **4× more likely** to be attacks than average.

**The big win in v2:**
Medium + High together capture **58.7% of all attacks** while covering only **36.5% of traffic.**
That means more than half of all attackers are being challenged or blocked,
while 63.5% of legitimate users pass through with zero friction.

---

### Rule-Level Stats

| Rule | Fires on | Attack Rate (triggered) | Attack Rate (not triggered) | Verdict |
|------|----------|-------------------------|-----------------------------|---------|
| rule_unusual_time      | 5.8%  | 18.6% | 8.6% |  Strong — 2× lift |
| rule_off_hours         | 7.3%  | 14.2% | 8.8% |  Moderate |
| rule_new_device        | 38.4% | 11.4% | 7.9% |  Mild but consistent |
| rule_new_asn           | 32.9% | 11.1% | 8.3% |  Mild but consistent |
| rule_recent_failures   | 36.7% | 12.5% | 7.3% |  Moderate |
| rule_risky_device_type | 0.13% | 58.6% | 9.2% |  Extremely strong — rare but deadly |
| rule_high_risk_country | 22.5% | 31.7% | 2.7% |  Dominant signal — 11× difference |

**Two standout rules added in v2:**

- **Risky device type** fires on almost nobody (0.13%) but when it does,
  58.6% of those logins are attacks. Nearly 1 in 2. Extremely high precision signal.

- **High risk country** is the biggest driver in the whole system.
  31.7% attack rate vs only 2.7% when not triggered — that's an 11× difference.
  This one rule alone reshapes the entire score distribution.

---

### V1 vs V2 — Quick Comparison

| Metric | V1 | V2 | Change |
|--------|----|----|--------|
| High band attack rate | 18.3% | 33.8% | ↑ Much more precise |
| High band size | 5.5% | 2.6% | ↓ Smaller but cleaner |
| Low band attack rate | 7.5% | 6.0% | ↓ Safer to allow |
| Attacks in medium+high | 48% | 58.7% | ↑ Catching more attackers |

V2 is strictly better. Smaller high band, higher attack rate inside it,
and a cleaner low band — all at the same time.
The two new rules (device type + country) did most of the heavy lifting.

Justifiaction:

Risky device type (bot/unknown):
	•	Extremely rare (~0.13% of events) but highly dangerous — almost 60% of those logins come from attack IPs.
	•	When this rule doesn’t fire, the attack rate drops back to 9.15%, close to global.
High-risk countries (ru, cl, ro, ru, us, vn, pk):
	•	About 22.5% of all logins, but with an attack rate of 31.7%,
	•	Compared to only 2.7% in all other countries.
	•	That’s more than 10× difference → clearly a dominant risk factor.

This absolutely justifies adding them as high-weight rules in v2.

# Conclusion

“We first built a simple rule-based baseline using time anomalies, device changes, ASN changes, and recent failure bursts. That baseline already separated risk into low, medium, and high bands, with the high band showing about twice the global attack rate.

In the second iteration, we incorporated two stronger contextual signals derived from our analysis: risky device types (such as ‘bot’ or ‘unknown’), and high-risk countries identified directly from the dataset. These rules are rare but highly predictive, so we assigned them higher weights in the risk score.

With the v2 rule baseline, the high-risk band now contains only 2.6% of all login attempts but has an attack rate of 33.8%, which is almost 4× the global average. At the same time, the low-risk band’s attack rate drops to about 6%, and the medium+high bands together capture nearly 59% of all attack IPs while covering only about 36% of the traffic. This shows that the enhanced rule-based system does a much better job of concentrating risky behavior, while keeping the low-risk band relatively clean and user-friendly.”

In [ ]:
label = "Is Attack IP"

print("\nAttack rate by decision (v2):")
print(df.groupby("rule_decision_v2")[label].mean())

print("\nCrosstab of decision vs label:")
print(pd.crosstab(df["rule_decision_v2"], df[label]))


Attack rate by decision (v2):
rule_decision_v2
ALLOW      0.059785
BLOCK      0.338107
STEP_UP    0.133906
Name: Is Attack IP, dtype: float64

Crosstab of decision vs label:
Is Attack IP       False  True 
rule_decision_v2               
ALLOW             179252  11398
BLOCK               5223   2668
STEP_UP            87873  13586


## V2 Decision Engine — Results

Every login gets one of three decisions: ALLOW, STEP_UP, or BLOCK.
Here's how well each decision aligns with reality.

---

### Attack Rate by Decision

| Decision | Logins  | Are Actually Attacks | Are Actually Legitimate |
|----------|---------|---------------------|------------------------|
| ALLOW    | 190,650 | 6.0%  (11,398)      | 94.0% (179,252)        |
| STEP_UP  | 101,459 | 13.4% (13,586)      | 86.6% (87,873)         |
| BLOCK    | 7,891   | 33.8% (2,668)       | 66.2% (5,223)          |

---

### The Good News

**ALLOW is mostly clean.**
94% of logins we let through are legitimate. Only 6% are attacks sneaking past us.
For a rule-based system with no ML, that's a reasonable low-friction zone.

**BLOCK is genuinely dangerous.**
1 in 3 logins we block is a real attack. That's 3.67× the global average.
The system isn't randomly blocking people — it's targeting a genuinely risky group.

**STEP_UP is doing its job.**
13.4% attack rate — higher than global (9.2%) but not extreme.
This is exactly the right zone for MFA. You're not sure enough to block,
but suspicious enough to ask for a second verification.

---

### False Negatives — The Security Risk

**11,398 attacks were told ALLOW. We missed them completely.**

These are your False Negatives (FN) — the system said "safe" but it was wrong.

In plain terms: real attackers walked right through the front door
because they didn't trigger enough rules. Maybe they logged in during
normal hours, used a known device, and had no recent failures.
They looked perfectly normal — and our rules had no way to know otherwise.

This is the most dangerous type of error in a security system.
A missed attacker can go on to take over an account, steal data,
or cause real damage. No friction was applied, no warning was raised.

This is the primary reason we move to an ML model next.
Rules can only catch what they were explicitly written for.
A learned model can find subtle patterns in combinations of signals
that no human would think to write a rule for.

---

### False Positives — The User Experience Cost

**5,223 legitimate users got blocked. We were wrong about them.**

These are your False Positives (FP) — the system said "risky" but it was wrong.

In plain terms: a real person, not an attacker, got blocked.
Maybe they were traveling in a high-risk country, logged in late at night,
and happened to be on a new device. Every rule fired — but it was just a
business trip, not an attack.

This is annoying and costly in a different way:
- Frustrated legitimate users
- Support tickets and password resets
- Loss of trust in the system

The STEP_UP group has an even larger false positive pool:
**87,873 legitimate users got asked for MFA unnecessarily.**
That's 87,873 people interrupted mid-login for something that wasn't their fault.

There's always a tradeoff here.
A tighter system catches more attackers (fewer FN) but annoys more
legitimate users (more FP). A looser system is friendlier but lets more
attacks through. The goal of the ML model is to find a smarter middle ground —
catching more real attacks while reducing unnecessary friction for honest users.

---

### Putting It All Together

| Error Type | Count | Who pays the price |
|------------|-------|--------------------|
| False Negative (missed attack) | 11,398 | The company / victim user |
| False Positive (wrongly blocked) | 5,223 | The legitimate user who got blocked |
| False Positive (wrongly stepped up) | 87,873 | The legitimate user who got MFA'd unnecessarily |

---

### One Line Summary

> "Our rule engine catches 58.7% of all attacks through STEP_UP and BLOCK decisions,
> but still misses 11,398 attacks (False Negatives) and incorrectly challenges
> 93,096 legitimate users (False Positives). This tradeoff between security
> and user experience is exactly what the ML model in the next notebook
> is designed to improve."

Predicted SAFE (ALLOW):

	•	Non-attack (True negative, TN): 179,252
	•	Attack (False negative, FN):   11,398

Predicted RISKY (STEP_UP + BLOCK):

Non-attack:

	•	STEP_UP: 87,873
	•	BLOCK:   5,223

→ False positives (FP) = 87,873 + 5,223 = 93,096

Attack:

	•	STEP_UP: 13,586
	•	BLOCK:   2,668
  
→ True positives (TP) = 13,586 + 2,668 = 16,254

In [ ]:
import pandas as pd
import numpy as np

# 1. Binary prediction from rule decisions
# 1 = risky (STEP_UP or BLOCK), 0 = safe (ALLOW)
df["pred_attack"] = df["rule_decision_v2"].isin(["STEP_UP", "BLOCK"]).astype(int)

y_true = df["Is Attack IP"].astype(int)   # 1 = attack, 0 = normal
y_pred = df["pred_attack"]

# 2. Confusion matrix components
TP = int(((y_pred == 1) & (y_true == 1)).sum())
FP = int(((y_pred == 1) & (y_true == 0)).sum())
TN = int(((y_pred == 0) & (y_true == 0)).sum())
FN = int(((y_pred == 0) & (y_true == 1)).sum())

total = TP + FP + TN + FN

print("Confusion counts:")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}, Total: {total}\n")

# 3. Metrics (with safe division)
def safe_div(num, den):
    return num / den if den != 0 else np.nan

accuracy  = safe_div(TP + TN, total)
precision = safe_div(TP, TP + FP)         # Positive Predictive Value
recall    = safe_div(TP, TP + FN)         # True Positive Rate (TPR, sensitivity)
specificity = safe_div(TN, TN + FP)       # True Negative Rate (TNR)
fpr      = safe_div(FP, FP + TN)          # False Positive Rate
fnr      = safe_div(FN, FN + TP)          # False Negative Rate
f1       = safe_div(2 * precision * recall, precision + recall)
balanced_accuracy = (recall + specificity) / 2

# 4. Put metrics into a nice table
metrics = {
    "Metric": [
        "Accuracy",
        "Precision (PPV)",
        "Recall / TPR (Sensitivity)",
        "Specificity / TNR",
        "FPR (1 - Specificity)",
        "FNR (Miss Rate)",
        "F1-score",
        "Balanced Accuracy",
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        specificity,
        fpr,
        fnr,
        f1,
        balanced_accuracy,
    ],
}

metrics_df = pd.DataFrame(metrics)
metrics_df["Value"] = metrics_df["Value"].round(4)

metrics_df

Confusion counts:
TP: 16254, FP: 93096, TN: 179252, FN: 11398, Total: 300000



,Metric,Value
0,Accuracy,0.6517
1,Precision (PPV),0.1486
2,Recall / TPR (Sensitivity),0.5878
3,Specificity / TNR,0.6582
4,FPR (1 - Specificity),0.3418
5,FNR (Miss Rate),0.4122
6,F1-score,0.2373
7,Balanced Accuracy,0.6230


## V2 Baseline — Performance Metrics

---

### Confusion Matrix

|                        | Predicted SAFE (ALLOW) | Predicted RISKY (STEP_UP / BLOCK) |
|------------------------|------------------------|-----------------------------------|
| **Actually Legitimate**|  TN = 179,252        |  FP = 93,096                    |
| **Actually Attack**    |  FN = 11,398         |  TP = 16,254                    |

---

### Metrics at a Glance

| Metric | Value | What it means in one line |
|--------|-------|---------------------------|
| Accuracy | 65.2% | Misleading here — dataset is imbalanced, ignore this |
| Precision | 14.9% | Only 1 in 7 logins we flag is actually an attack |
| Recall | 58.8% | We catch 6 out of every 10 real attacks |
| Specificity | 65.8% | 2 in 3 legit users pass through without friction |
| FPR | 34.2% | 1 in 3 legit users gets unnecessarily challenged |
| FNR | 41.2% | 4 in 10 real attacks slip through undetected |
| F1-Score | 23.7% | Low — lots of false alarms hurting our precision |
| Balanced Accuracy | 62.3% | Moderate — better than random (50%), room to grow |

---

### The Core Tradeoff
```
         We MISS 4 in 10 attacks          We CATCH 6 in 10 attacks
                                                  
         FN = 11,398                       TP = 16,254
         (slipped through)                 (correctly flagged)


         We WRONGLY flag 1 in 3 legit      We CORRECTLY clear 2 in 3 legit
                                                  
         FP = 93,096                       TN = 179,252
         (unnecessary friction)            (smooth experience)
```

---

### Two Problems, One Root Cause

**Problem 1 — Security gap (FNR 41%)**
4 in 10 attackers look "normal" by our rules and walk straight through.
Fixed rules can't catch what they weren't written for.

**Problem 2 — Friction cost (FPR 34%)**
1 in 3 legitimate users gets challenged or blocked unnecessarily.
Rules don't understand context — a business traveller looks the same as an attacker.

Both problems exist because **rules don't learn from data.**
They treat every login the same way given the same score,
regardless of subtle patterns a model could pick up.

> This is exactly the gap the ML classifier in the next notebook is built to close.

In [ ]:
confusion_matrix_df = pd.DataFrame(
    {
        "Actual Normal (0)": [TN, FP],
        "Actual Attack (1)": [FN, TP],
    },
    index=["Predicted SAFE (0)", "Predicted RISKY (1)"],
)

confusion_matrix_df["Row Total"] = confusion_matrix_df.sum(axis=1)
confusion_matrix_df.loc["Column Total"] = confusion_matrix_df.sum(axis=0)

confusion_matrix_df

,Actual Normal (0),Actual Attack (1),Row Total
Predicted SAFE (0),179252,11398,190650
Predicted RISKY (1),93096,16254,109350
Column Total,272348,27652,300000


1️Accuracy (0.6517 → ~65%)

“How often is the system correct overall?”

	•	About 65% of all logins are classified correctly:
	•	either normal and treated as safe
	•	or attack and treated as risky
 In imbalanced problems like this (only ~9% attacks), accuracy is not the best metric, but okay to report.

⸻

2️Precision / PPV (0.1486 → ~14.9%)

“When we say a login is risky (STEP_UP/BLOCK), how often is it actually an attack?”

	•	Only about 15% of the logins we flag as risky are truly attacks.
	•	That means 85% of risky decisions are false positives (legit users being challenged/blocked).

Interpretation:
	•	The system is paranoid: it flags a lot of logins as risky.
	•	Good for security, but high friction for users.

⸻

3️Recall / TPR / Sensitivity (0.5878 → ~58.8%)

“Out of all attack logins, how many do we catch as risky?”

	•	Your system correctly flags about 59% of attacks.
	•	The other 41% of attacks slip into ALLOW (these are FNs).

Interpretation:
	•	Your baseline catches more than half of attacks.
	•	But still misses a large chunk, which is exactly why you want to improve using ML.

⸻

4️Specificity / TNR (0.6582 → ~65.8%)

“Out of all normal (non-attack) logins, how many do we correctly treat as safe?”

	•	About 66% of normal logins are correctly allowed.
	•	That means 34% of normal logins are incorrectly treated as risky (this is the FPR).

 Interpretation:
	•	One third of legitimate users get some extra friction (STEP_UP or BLOCK).
	•	This confirms the system is conservative and noisy.

⸻

5️ FPR (0.3418 → ~34.2%)

False Positive Rate = ‘How many legit logins do we annoy?’

	•	Among all normal logins, 34% are flagged as risky.
	•	In cybersecurity, this is what leads to alert fatigue and frustrated users.

⸻

6️ FNR (0.4122 → ~41.2%)

False Negative Rate = ‘How many attacks do we miss?’

	•	About 41% of attacks are treated as safe (they get ALLOW).
	•	This is dangerous from a security perspective.

⸻

7️ F1-score (0.2373 → ~0.24)

“Balance between precision and recall for attacks.”

	•	F1 is low (~0.24), which reflects:
	•	okay recall (catching ~59% of attacks),
	•	but poor precision (lots of false alarms).

Interpretation:
	•	The baseline finds many attacks, but not very cleanly (lots of noise).

⸻

8️ Balanced Accuracy (0.6230 → ~62.3%)

“Average of: how good we are on attacks (recall) and how good we are on normal logins (specificity).”

\text{Balanced Accuracy} = \frac{TPR + TNR}{2} = \frac{0.5878 + 0.6582}{2} ≈ 0.623

	•	This metric treats attack and normal classes equally important.
	•	~62% is moderate performance.

  •	Time-based: is_off_hours, is_unusual_time_for_user, user_hour_std_q
  
	•	Behavior: new_device_flag, new_asn_flag, failures_last_5, burst_failure_count
	•	Rarity: location_rarity_q, device_type_rarity_q, asn_rarity_q
	•	Geo/device: high-risk countries, risky device types

In [ ]:
df.head()

,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,...,rule_new_asn,rule_recent_failures,rule_risk_score,rule_risk_band,rule_risky_device_type,rule_high_risk_country,rule_risk_score_v2,rule_risk_band_v2,rule_decision_v2,pred_attack
0,2020-02-06 17:10:54.364,-9223287066183308537,541.0,84.209.76.159,no,oslo county,oslo,41164,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...,Chrome 69.0.3497.17.19,...,0,0,0,low,0,0,0,low,ALLOW,0
1,2020-02-06 19:52:41.530,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0,0,0,low,0,0,0,low,ALLOW,0
2,2020-02-06 20:55:19.627,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0,0,0,low,0,0,0,low,ALLOW,0
3,2020-02-05 21:03:20.657,-9223200578825105501,541.0,79.161.56.83,no,vestfold og telemark,holmestrand,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Chrome Mobile 81.0.4044.2033,...,0,0,0,low,0,0,0,low,ALLOW,0
4,2020-02-06 19:12:29.501,-9223199305075633823,541.0,79.161.86.86,no,-,-,29695,Mozilla/5.0 (Linux; U; Android 13.0; i phone X...,Opera Mobile 52.1.2254,...,0,0,0,low,0,0,0,low,ALLOW,0


In [ ]:
print("Shape:", df.shape)
df.columns.tolist()

Shape: (300000, 70)


['Login Timestamp',
 'User ID',
 'Round-Trip Time [ms]',
 'IP Address',
 'Country',
 'Region',
 'City',
 'ASN',
 'User Agent String',
 'Browser Name and Version',
 'OS Name and Version',
 'Device Type',
 'Login Successful',
 'Is Attack IP',
 'Is Account Takeover',
 'browser',
 'os',
 'hour',
 'dayofweek',
 'is_new_device_for_user',
 'is_new_ip_for_user',
 'is_off_hours',
 'failed_login',
 'failures_last_5',
 'failure_streak',
 'failure_streak_capped',
 'location',
 'new_location_flag',
 'new_asn_flag',
 'device_fingerprint',
 'valid_device',
 'new_device_flag',
 'device_change_rate',
 'ts_sec',
 'delta_sec',
 'new_window',
 'window_id',
 'logins_5min',
 'failure_flag',
 'burst_failure_count',
 'failure_rate',
 'streak_reset',
 'streak_id',
 'failure_streak_length',
 'location_freq',
 'location_rarity',
 'device_type_freq',
 'device_type_rarity',
 'asn_freq',
 'asn_rarity',
 'location_rarity_q',
 'device_type_rarity_q',
 'asn_rarity_q',
 'user_offhour_rate',
 'is_unusual_time_for_user',

In [ ]:
# # Choose only the columns you actually want to carry forward.
# # For now, let's just save everything in df.
# # (You can refine later if needed.)

# save_path_parquet = "./data/rule_baseline__v1+v2.parquet"
# save_path_csv = "./data/rule_baseline__v1+v2.csv"

# # Parquet (preferred: smaller, preserves dtypes)
# df.to_parquet(save_path_parquet, index=False)

# # Optional CSV backup
# df.to_csv(save_path_csv, index=False)

# print("Saved feature dataset to:")
# print(" -", save_path_parquet)
# print(" -", save_path_csv)
# print("Shape:", df.shape)